# Sokoban Pathfinding Prototype

This notebook explores the grid-navigation component of a Sokoban-style game. It provides manually defined and randomly generated grids, an interactive player, and implementations of Random Search, Depth-First Search (DFS), Breadth-First Search (BFS), and A*.

> **Scope:** this prototype focuses on navigating a player through static walls from a start cell to a goal cell. Box-pushing mechanics are intentionally outside its current scope.

The notebook is structured as follows:

- **Configuration**
  - Libraries
  - Grids
  - Game logic
- **Manual play**
- **Search algorithms and evaluation**


# Configuration

#### Libraries

In [1]:
import time
import random

import numpy as np

from IPython.display import display, clear_output
import ipywidgets as widgets

import collections

import heapq
import itertools

#### Grids

**Manually defined grids**
- `0`: empty block
- `1`: start
- `2`: end
- `3`: wall

In [2]:
GRIDS = [

    # Level 0
    [[1, 0],
     [0, 2]],

    # Level 1
    [[1, 0, 0],
     [0, 0, 0],
     [0, 0, 2]],

    # Level 2
    [[0, 0, 0, 1],
     [0, 0, 0, 0],
     [0, 0, 0, 0],
     [2, 0, 0, 0]],

    # Level 3
    [[1, 3, 0, 0, 0],
     [0, 0, 0, 0, 0],
     [0, 0, 0, 0, 0],
     [0, 3, 0, 0, 3],
     [0, 3, 0, 0, 2]],

    # Level 4
    [[0, 3, 1, 0, 0, 0],
     [0, 0, 0, 0, 0, 3],
     [0, 0, 0, 3, 0, 0],
     [0, 3, 3, 3, 3, 0],
     [0, 0, 0, 0, 2, 0],
     [3, 3, 3, 0, 3, 0]],

    # Level 5
    [[1, 0, 0, 0, 0, 3, 0],
     [0, 3, 3, 3, 0, 0, 0],
     [0, 0, 0, 3, 0, 3, 0],
     [3, 3, 0, 0, 3, 0, 0],
     [0, 0, 0, 3, 0, 0, 0],
     [3, 0, 3, 3, 3, 3, 0],
     [3, 0, 0, 0, 2, 3, 0]],

    # Level 6
    [[0, 0, 0, 3, 0, 0, 0, 0],
     [0, 3, 0, 0, 0, 3, 0, 3],
     [0, 0, 0, 3, 3, 3, 0, 0],
     [0, 3, 0, 3, 0, 0, 3, 0],
     [0, 0, 3, 0, 0, 3, 0, 2],
     [3, 0, 0, 3, 0, 3, 3, 3],
     [0, 3, 0, 0, 0, 3, 0, 0],
     [1, 0, 0, 3, 0, 0, 0, 0]],

    # Level 7
    [[3, 0, 0, 0, 0, 3, 0, 0, 0],
     [0, 3, 0, 3, 0, 0, 0, 3, 0],
     [0, 0, 0, 3, 3, 3, 0, 0, 0],
     [0, 0, 3, 3, 0, 0, 3, 0, 0],
     [3, 0, 0, 3, 1, 3, 0, 0, 3],
     [0, 0, 3, 3, 0, 0, 3, 0, 0],
     [0, 3, 0, 3, 0, 3, 0, 3, 0],
     [0, 3, 0, 0, 0, 3, 0, 3, 0],
     [0, 0, 0, 0, 3, 2, 0, 0, 0]],

    # Level 8
    [[1, 0, 0, 0, 3, 0, 0, 0, 0, 0],
     [0, 3, 3, 0, 3, 0, 3, 3, 0, 0],
     [0, 0, 3, 0, 3, 0, 0, 0, 3, 0],
     [3, 0, 3, 0, 0, 0, 0, 3, 0, 0],
     [0, 0, 3, 0, 3, 3, 3, 3, 3, 0],
     [0, 0, 0, 0, 3, 0, 0, 0, 3, 0],
     [0, 3, 3, 0, 3, 2, 3, 0, 0, 0],
     [0, 3, 0, 0, 3, 3, 3, 3, 0, 0],
     [0, 0, 0, 3, 3, 0, 0, 0, 0, 0],
     [3, 3, 0, 0, 0, 0, 3, 0, 0, 3]]

]

**Random grid generator**

In [3]:
def grid_generator(size=(5,5), seed=None, wall_ratio=0.2):
    '''
    Generates a grid with the specified size, start, end, and random walls.

    Attributes:
    - size: tuple, size of the grid
    - seed: int, random seed
    - wall_ratio: float, portion of the grid to be filled with walls

    Returns:
    - grid: numpy array

    '''

    # Create grid of zeros
    grid = np.zeros(size, dtype=int)

    # Optional reproducibility
    if seed is not None:
        np.random.seed(seed)

    num_cells = np.prod(size)

    # 1. Sample start and end positions
    flat_indices = np.random.choice(num_cells, size=2, replace=False)
    row_idx, col_idx = np.unravel_index(flat_indices, size)

    start = (row_idx[0], col_idx[0])
    end = (row_idx[1], col_idx[1])

    grid[start] = 1
    grid[end] = 2

    # 2. Generate random walls in remaining empty spaces
    # Get indices of all cells that are still 0
    empty_indices = np.argwhere(grid == 0)
    num_walls = int(len(empty_indices) * wall_ratio)

    if num_walls > 0:
        wall_selection = np.random.choice(len(empty_indices), size=num_walls, replace=False)
        for idx in wall_selection:
            grid[tuple(empty_indices[idx])] = 3

    return grid

#### Logic

**Configuration**

In [4]:
MOVES = {
    "up": (-1, 0),
    "down": (1, 0),
    "left": (0, -1),
    "right": (0, 1),
}

SYMBOLS = {
    0: "   ",
    1: " s ",
    2: " E ",
    3: "[X]"
}

**Game logic**

In [5]:
def get_start(grid):
    '''
    Gets the position of the starting block.

    Arguments:
    - grid: numpy array

    Returns:
    - tuple, position of the start

    '''
    start_x, start_y = np.argwhere(grid == 1)[0]
    return (int(start_x), int(start_y))


def is_win(grid, position):
    '''
    Checks whether the player has reached the end block and stops the game.

    Arguments:
    - grid: numpy array
    - position: tuple, current position of the player

    Returns:
    - boolean (only if manual_=False), whether the player has won

    '''

    if grid[position] == 2:
        if manual_:
            with output:
                print("Reached goal!")

            # Disable buttons
            btn_up.disabled = True
            btn_down.disabled = True
            btn_left.disabled = True
            btn_right.disabled = True
        else:
            return True
    elif not manual_:
        return False

def step(grid, position, action):
    '''
    Moves the player by one step, considering walls.

    Arguments:
    - position: tuple, current position of the player.
    - action: string, legal player move
    - grid: numpy array.

    Returns:
    - tuple, updated position

    '''

    # Calculate new position
    i, j = position
    di, dj = MOVES[action]
    new_i, new_j = i + di, j + dj

    # Check grid boundaries
    if 0 <= new_i < grid.shape[0] and 0 <= new_j < grid.shape[1]:
        # Check for walls (value 3)
        if grid[new_i, new_j] == 3:
            return position
        return (new_i, new_j)
    else:
        return position


def move(action):
    '''
    Moves the player in the desired direction and displays the result.

    Arguments:
    - action: string, legal player move.

    '''

    # Check if the direction is legal
    if action not in MOVES:
        raise ValueError("Invalid move")

    # Update position
    global position
    position = step(grid_, position, action)

    # Render output
    render()

    # If playing manually, check if player won
    if manual_:
        is_win(grid_, position)


def display_grid(grid, position):
    '''
    Displays the current status of the game.

    Arguments:
    - grid: numpy array
    - position: tuple, current position of the player

    '''
    rows, cols = grid.shape[0], grid.shape[1]
    # Ensure 3 is in symbols for walls
    local_symbols = SYMBOLS.copy()
    if 3 not in local_symbols: local_symbols[3] = " # "

    print("\n+" + "---+" * cols)
    for i in range(rows):
        row_str = "|"
        for j in range(cols):
            if (i, j) == position:
                row_str += " P |"
            else:
                row_str += local_symbols.get(grid[i][j], " ? ") + "|"
        print(row_str)
        print("+" + "---+" * cols)


def render():
    '''Displays the grid.'''
    if manual_:
        with output:
            clear_output(wait=True)
            display_grid(grid_, position)
    else:
        clear_output(wait=True)
        display_grid(grid_, position)


def display_path(grid, path, first_visit=False, start_index=0):
    '''
    Displays the path taken by the algorithm.
    It displays the grid with an iteration number in each visited cell.

    Arguments:
    - grid: numpy array
    - path: list of tuples in the order visited
    - first_visit: if True annotate the first time a cell was visited (else last)
    - start_index: index to start numbering from (0 in our case for simplicity)
    '''

    normalized = [tuple(p) for p in path]
    iter_map = {}

    for k, pos in enumerate(normalized, start=start_index):
        if first_visit:
            iter_map.setdefault(pos, k)
        else:
            iter_map[pos] = k

    rows, cols = grid.shape
    local_symbols = SYMBOLS.copy()
    if 3 not in local_symbols: local_symbols[3] = " # "

    print("\n+" + "---+" * cols)
    for i in range(rows):
        row_str = "|"
        for j in range(cols):
            pos = (i, j)
            if pos in iter_map:
                row_str += f"{iter_map[pos]:3d}|"
            else:
                row_str += local_symbols.get(grid[i][j], "   ") + "|"
        print(row_str)
        print("+" + "---+" * cols)


def play(grid, manual):
    '''
    Sets up the game.
    '''
    global grid_
    grid_ = np.array(grid)

    global manual_
    manual_ = manual

    # Set the starting player position
    start = get_start(grid_)
    global position
    position = start

    if manual:
        global output
        output = widgets.Output()

        global btn_up, btn_down, btn_left, btn_right
        btn_up = widgets.Button(description="↑")
        btn_down = widgets.Button(description="↓")
        btn_left = widgets.Button(description="←")
        btn_right = widgets.Button(description="→")

        btn_up.on_click(lambda _: move("up"))
        btn_down.on_click(lambda _: move("down"))
        btn_left.on_click(lambda _: move("left"))
        btn_right.on_click(lambda _: move("right"))

        controls = widgets.VBox([
            widgets.HBox([widgets.Label(value="Controls:"), btn_up]),
            widgets.HBox([btn_left, btn_down, btn_right]),
        ])
        display(output, controls)

    render()

**Algorithm training**

In [6]:
def evaluate_algorithm(algorithm, grid, delay=0.5):
    '''
    Function to evaluate algorithms.

    Arguments:
    - algorithm: function, algorithm to be evaluated
    - grid: numpy array
    - delay: time between two iterations.

    '''

    # Start game
    play(grid, manual=False)

    # Initialize counter variables
    iterations = 0
    actions = []
    path = [position]

    # Run algorithm until game ends
    while True:
        time.sleep(delay)

        # Move player
        action = algorithm(grid, position)

        if action is None:
            clear_output(wait=True)
            display_path(grid, path)
            print("No solution was found for this level.")
            return None

        move(action)
        print(action)

        # Update counters
        iterations += 1
        actions.append(action)
        path.append(tuple(int(x) for x in position))

        # Check if target has been reached
        if is_win(grid, position):
            break

        if iterations >= 100000:
            clear_output(wait=True)
            display_path(grid, path)
            print(f"Iterations limit reached. The algorithm couldn't solve this level")
            return

    # Print results
    clear_output(wait=True)
    display_path(grid, path)
    print(f"\nSolved in {iterations} steps by {algorithm.__name__}.\n")
    return iterations

# Manual

Create random grid with the desired size:

In [7]:
random_grid = grid_generator(size=(5,5))

or play with a manually defined grid:

In [8]:
grid_number = 3

manual_grid = np.array(GRIDS[grid_number])

When you're ready to play, run this cell:

In [9]:
play(grid=manual_grid, manual=True)


+---+---+---+---+---+
| P |[X]|   |   |   |
+---+---+---+---+---+
|   |   |   |   |   |
+---+---+---+---+---+
|   |   |   |   |   |
+---+---+---+---+---+
|   |[X]|   |   |[X]|
+---+---+---+---+---+
|   |[X]|   |   | E |
+---+---+---+---+---+


# Algorithms

Define the algorithm in the cell below.

The algorithm should:
- accept **grid** and **position** as input;
- return an **action**.

#### Random

In [10]:
def random_algorithm(grid, position):
    return random.choice(list(MOVES.keys()))

In [11]:
random.seed(42)

for i in range(len(GRIDS)):
    level = np.array(GRIDS[i])
    evaluate_algorithm(random_algorithm, level, delay=0.0)


+---+---+---+---+---+---+---+---+---+---+
|419|401|394|391|[X]|1062|1065|1131|1136|1139|
+---+---+---+---+---+---+---+---+---+---+
|422|[X]|[X]|387|[X]|1031|[X]|[X]|1144|1145|
+---+---+---+---+---+---+---+---+---+---+
|705|706|[X]|386|[X]|1030|1023|1022|[X]|1146|
+---+---+---+---+---+---+---+---+---+---+
|[X]|976|[X]|1012|1015|1029|1028|[X]|1148|1149|
+---+---+---+---+---+---+---+---+---+---+
|996|995|[X]|1010|[X]|[X]|[X]|[X]|[X]|1155|
+---+---+---+---+---+---+---+---+---+---+
|997|998|1002|1009|[X]|1230|1229|1226|[X]|1220|
+---+---+---+---+---+---+---+---+---+---+
|967|[X]|[X]|929|[X]|1231|[X]|1225|1223|1222|
+---+---+---+---+---+---+---+---+---+---+
|799|[X]|920|921|[X]|[X]|[X]|[X]|1190|1187|
+---+---+---+---+---+---+---+---+---+---+
|904|917|919|[X]|[X]|   |   |   |1179|1165|
+---+---+---+---+---+---+---+---+---+---+
|[X]|[X]|913|888|867|859|[X]|   |1178|[X]|
+---+---+---+---+---+---+---+---+---+---+

Solved in 1231 steps by random_algorithm.



#### Depth-first search

In [12]:
def make_DFS():
    stack = None
    visited = None
    solution_path = None
    initialized = False

    def DFS(grid, position):
        nonlocal stack, visited, solution_path, initialized

        # initialize once per new evaluation
        if not initialized:
            stack = [(position, [])]
            visited = set()
            solution_path = None
            initialized = True

        # if we already found the solution, return next move
        if solution_path is not None:
            return solution_path.pop(0) if solution_path else None

        # continue DFS exploration
        while stack:
            current_pos, path = stack.pop()

            if current_pos in visited:
                continue

            visited.add(current_pos)

            # goal check
            if grid[current_pos] == 2:
                solution_path = path
                return solution_path.pop(0) if solution_path else None

            # expand neighbors
            for action, (di, dj) in MOVES.items():
                new_i = current_pos[0] + di
                new_j = current_pos[1] + dj
                new_pos = (new_i, new_j)

                if (0 <= new_i < grid.shape[0] and
                    0 <= new_j < grid.shape[1] and
                    grid[new_i, new_j] != 3):

                    if new_pos not in visited:
                        stack.append((new_pos, path + [action]))

        return None  # no solution

    return DFS

In [13]:
for i in range(len(GRIDS)):
    level = np.array(GRIDS[i])

    DFS = make_DFS()  # new instance per level
    evaluate_algorithm(DFS, level, delay=0.0)



+---+---+---+---+---+---+---+---+---+---+
|  0|  1|  2|  3|[X]| 13| 14| 15| 16| 17|
+---+---+---+---+---+---+---+---+---+---+
|   |[X]|[X]|  4|[X]| 12|[X]|[X]|   | 18|
+---+---+---+---+---+---+---+---+---+---+
|   |   |[X]|  5|[X]| 11| 10|   |[X]| 19|
+---+---+---+---+---+---+---+---+---+---+
|[X]|   |[X]|  6|  7|  8|  9|[X]|   | 20|
+---+---+---+---+---+---+---+---+---+---+
|   |   |[X]|   |[X]|[X]|[X]|[X]|[X]| 21|
+---+---+---+---+---+---+---+---+---+---+
|   |   |   |   |[X]| 28| 27| 26|[X]| 22|
+---+---+---+---+---+---+---+---+---+---+
|   |[X]|[X]|   |[X]| 29|[X]| 25| 24| 23|
+---+---+---+---+---+---+---+---+---+---+
|   |[X]|   |   |[X]|[X]|[X]|[X]|   |   |
+---+---+---+---+---+---+---+---+---+---+
|   |   |   |[X]|[X]|   |   |   |   |   |
+---+---+---+---+---+---+---+---+---+---+
|[X]|[X]|   |   |   |   |[X]|   |   |[X]|
+---+---+---+---+---+---+---+---+---+---+

Solved in 29 steps by DFS.



#### Breadth-first search

In [14]:
def BFS(grid, position):
    "Finds the shortest path to the end using Breadth-first search (BFS)"

    if grid[position] == 2:
        return None #already on the end

    queue = collections.deque() #double ended queue
    queue.append((position, [])) #pos , list of the actions to get there (path)

    visited = set() #tracks visited pos
    visited.add(position)

    while queue:
        current_pos, path = queue.popleft()

        if grid[current_pos] == 2: #already reached end
            return path[0]

        #check neighbouring pos
        for action, (di, dj) in MOVES.items():
            new_i = current_pos[0] + di
            new_j = current_pos[1] + dj
            new_pos = (new_i, new_j)

            if (0 <= new_i < grid.shape[0]
                and 0 <= new_j < grid.shape[1]
                and grid[new_i,new_j] != 3):

                if new_pos not in visited:
                    visited.add(new_pos)

                    #update
                    new_path = list(path)
                    new_path.append(action) #add the steps to the path

                    queue.append((new_pos, new_path))

In [15]:
for i in range(len(GRIDS)):
    level = np.array(GRIDS[i])
    evaluate_algorithm(BFS, level, delay=0.0)


+---+---+---+---+---+---+---+---+---+---+
|  0|   |   |   |[X]|   |   |   |   |   |
+---+---+---+---+---+---+---+---+---+---+
|  1|[X]|[X]|   |[X]|   |[X]|[X]|   |   |
+---+---+---+---+---+---+---+---+---+---+
|  2|  3|[X]|   |[X]|   |   |   |[X]|   |
+---+---+---+---+---+---+---+---+---+---+
|[X]|  4|[X]|   |   |   |   |[X]|   |   |
+---+---+---+---+---+---+---+---+---+---+
|   |  5|[X]|   |[X]|[X]|[X]|[X]|[X]|   |
+---+---+---+---+---+---+---+---+---+---+
|  7|  6|   |   |[X]| 26| 25| 24|[X]|   |
+---+---+---+---+---+---+---+---+---+---+
|  8|[X]|[X]|   |[X]| 27|[X]| 23| 22|   |
+---+---+---+---+---+---+---+---+---+---+
|  9|[X]|   |   |[X]|[X]|[X]|[X]| 21|   |
+---+---+---+---+---+---+---+---+---+---+
| 10| 11| 12|[X]|[X]| 17| 18| 19| 20|   |
+---+---+---+---+---+---+---+---+---+---+
|[X]|[X]| 13| 14| 15| 16|[X]|   |   |[X]|
+---+---+---+---+---+---+---+---+---+---+

Solved in 27 steps by BFS.



#### A*

In [16]:
def get_goal(grid):
    for row in range(len(grid)):
        for col in range(len(grid[0])):
            if grid[row][col] == 2:
                return (row, col)

    return None

In [17]:
# heuristic
def manhattan_distance(pos1, pos2):
    return abs(pos1[0] - pos2[0]) + abs(pos1[1] - pos2[1])

In [18]:
def A_star(grid, position):
    "Finds the shortest path to the end using A* Search"

    if grid[position] == 2:
        return None

    goal = get_goal(grid)
    pq = [] #priority queue
    counter = itertools.count() #ensures FIFO and avoids crashing if f_score is the same for multiple tuples

    #insert in pq (but keep sorted)
    heapq.heappush(pq, (0, next(counter), position, [])) #inital state: f_score=0, counter, pos, path

    g_scores = {position: 0} #keeps track of the lowest path to ANY tile visited

    while pq:
        current_f, c, current_pos, path = heapq.heappop(pq) # Pop the state with the lowest f_score

        if grid[current_pos] == 2: #already reached goal
            return path[0]

        for action, (di, dj) in MOVES.items():
            new_i = current_pos[0] + di
            new_j = current_pos[1] + dj
            new_pos = (new_i, new_j)

            if (0 <= new_i < grid.shape[0]
                and 0 <= new_j < grid.shape[1]
                and grid[new_i, new_j] != 3):

                tentative_g_score = g_scores[current_pos] + 1  #current path cost + 1 step

                #if shortcut is found, evaluate node again
                if tentative_g_score < g_scores.get(new_pos, float('inf')):
                    g_scores[new_pos] = tentative_g_score # new best g_score
                    h_score = manhattan_distance(new_pos, goal) #estimated guess
                    f_score = tentative_g_score + h_score # f = g + h, tot cost

                    #update
                    new_path = list(path)
                    new_path.append(action)

                    heapq.heappush(pq, (f_score, next(counter), new_pos, new_path))

In [19]:
for i in range(len(GRIDS)):
    level = np.array(GRIDS[i])
    evaluate_algorithm(A_star, level, delay=0.0)


+---+---+---+---+---+---+---+---+---+---+
|  0|   |   |   |[X]|   |   |   |   |   |
+---+---+---+---+---+---+---+---+---+---+
|  1|[X]|[X]|   |[X]|   |[X]|[X]|   |   |
+---+---+---+---+---+---+---+---+---+---+
|  2|  3|[X]|   |[X]|   |   |   |[X]|   |
+---+---+---+---+---+---+---+---+---+---+
|[X]|  4|[X]|   |   |   |   |[X]|   |   |
+---+---+---+---+---+---+---+---+---+---+
|   |  5|[X]|   |[X]|[X]|[X]|[X]|[X]|   |
+---+---+---+---+---+---+---+---+---+---+
|   |  6|  7|  8|[X]| 26| 25| 24|[X]|   |
+---+---+---+---+---+---+---+---+---+---+
|   |[X]|[X]|  9|[X]| 27|[X]| 23| 22|   |
+---+---+---+---+---+---+---+---+---+---+
|   |[X]| 11| 10|[X]|[X]|[X]|[X]| 21|   |
+---+---+---+---+---+---+---+---+---+---+
|   |   | 12|[X]|[X]| 17| 18| 19| 20|   |
+---+---+---+---+---+---+---+---+---+---+
|[X]|[X]| 13| 14| 15| 16|[X]|   |   |[X]|
+---+---+---+---+---+---+---+---+---+---+

Solved in 27 steps by A_star.



## Algorithm comparison

The table below evaluates DFS, BFS, and A* on every predefined level without animation. Each value is the number of moves required to reach the goal. BFS and A* are expected to return shortest paths on these unweighted grids, while DFS prioritizes depth and may therefore produce longer routes.


In [20]:
def benchmark_algorithm(algorithm_factory, grid, max_steps=100000):
    """Returns the number of moves required to solve a grid, or None."""
    level = np.array(grid)
    current_position = get_start(level)
    algorithm = algorithm_factory()

    for steps in range(max_steps + 1):
        if is_win(level, current_position):
            return steps

        action = algorithm(level, current_position)
        if action is None:
            return None

        current_position = step(level, current_position, action)

    return None


algorithm_factories = [
    ("DFS", make_DFS),
    ("BFS", lambda: BFS),
    ("A*", lambda: A_star),
]

print("| Level | Grid size | DFS | BFS | A* |")
print("|---:|:---:|---:|---:|---:|")

for level_number, level in enumerate(GRIDS):
    results = [
        benchmark_algorithm(factory, level)
        for _, factory in algorithm_factories
    ]
    rows, columns = np.array(level).shape
    print(
        f"| {level_number} | {rows} x {columns} | "
        + " | ".join(str(result) for result in results)
        + " |"
    )


| Level | Grid size | DFS | BFS | A* |
|---:|:---:|---:|---:|---:|
| 0 | 2 x 2 | 2 | 2 | 2 |
| 1 | 3 x 3 | 8 | 4 | 4 |
| 2 | 4 x 4 | 12 | 6 | 6 |
| 3 | 5 x 5 | 12 | 8 | 8 |
| 4 | 6 x 6 | 14 | 8 | 8 |
| 5 | 7 x 7 | 12 | 12 | 12 |
| 6 | 8 x 8 | 22 | 22 | 22 |
| 7 | 9 x 9 | 39 | 35 | 35 |
| 8 | 10 x 10 | 29 | 27 | 27 |


Across the nine predefined levels, BFS and A* consistently find paths with the same minimum number of moves. DFS also solves every level, but its depth-first traversal produces longer paths on several grids. A* combines optimal paths with goal-directed exploration through the Manhattan-distance heuristic.
